In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
df = pd.read_csv('shopping_trends.csv')

In [42]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Payment Method,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Preferred Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Credit Card,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Bank Transfer,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Cash,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,PayPal,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Cash,Free Shipping,Yes,Yes,31,PayPal,Annually


In [43]:
## need to remove the columns 
remove = ['Customer ID','Location','Promo Code Used','Color','Subscription Status','Payment Method','Frequency of Purchases']
df.drop(columns=remove,axis=1,inplace=True)

In [44]:
df.shape

(3900, 12)

In [45]:
df.isna().sum()

Age                         0
Gender                      0
Item Purchased              0
Category                    0
Purchase Amount (USD)       0
Size                        0
Season                      0
Review Rating               0
Shipping Type               0
Discount Applied            0
Previous Purchases          0
Preferred Payment Method    0
dtype: int64

### Based on customer input our aim is to create a model which can predict what item the customer need to purchase 

In [46]:
### get all numerical columns and categorical columns separately
cat_features = [col for col in df.columns if df[col].dtype=='O']
num_features = [col for col in df.columns if df[col].dtype!='O']

In [47]:
cat_features

['Gender',
 'Item Purchased',
 'Category',
 'Size',
 'Season',
 'Shipping Type',
 'Discount Applied',
 'Preferred Payment Method']

In [48]:
num_features

['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']

In [49]:
for col in cat_features:
    print(df[col].value_counts())
    print('*'*35)

Gender
Male      2652
Female    1248
Name: count, dtype: int64
***********************************
Item Purchased
Blouse        171
Pants         171
Jewelry       171
Shirt         169
Dress         166
Sweater       164
Jacket        163
Coat          161
Sunglasses    161
Belt          161
Sandals       160
Socks         159
Skirt         158
Scarf         157
Shorts        157
Hat           154
Handbag       153
Hoodie        151
Shoes         150
T-shirt       147
Sneakers      145
Boots         144
Backpack      143
Gloves        140
Jeans         124
Name: count, dtype: int64
***********************************
Category
Clothing       1737
Accessories    1240
Footwear        599
Outerwear       324
Name: count, dtype: int64
***********************************
Size
M     1755
L     1053
S      663
XL     429
Name: count, dtype: int64
***********************************
Season
Spring    999
Fall      975
Winter    971
Summer    955
Name: count, dtype: int64
***********************

### Label Encoding on : Item Purchased column
### Remaining all : One Hot Encoding
### Scaling : Standard Schlar

In [61]:
X= df.drop('Item Purchased',axis=1)
y= df['Item Purchased']


In [62]:
cat_features = X.select_dtypes(include='O').columns
num_features = X.select_dtypes(exclude='O').columns

In [63]:
from sklearn.model_selection import train_test_split 
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=42)

In [64]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier

In [65]:
preprocessor = ColumnTransformer(
        [
            ('OneHotEncoder',OneHotEncoder(drop='first'),cat_features),
            ('StandardScaler',StandardScaler(),num_features)
]
    )

In [66]:
X_train = preprocessor.fit_transform(X_train)

In [67]:
X_train

array([[ 1.        ,  1.        ,  0.        , ...,  0.24740506,
         0.91396793, -0.78537487],
       [ 1.        ,  0.        ,  0.        , ...,  0.03620232,
        -0.63435032,  1.00542818],
       [ 1.        ,  1.        ,  0.        , ...,  1.04997547,
         0.91396793, -1.47414527],
       ...,
       [ 1.        ,  0.        ,  1.        , ...,  1.13445657,
         0.49169932, -1.54302231],
       [ 0.        ,  0.        ,  0.        , ...,  0.20516451,
         0.63245552,  0.5232889 ],
       [ 0.        ,  1.        ,  0.        , ..., -0.34396262,
         0.0694307 ,  0.66104298]], shape=(3120, 25))

In [68]:
X_test = preprocessor.transform(X_test)

In [69]:
X_test

array([[ 1.        ,  1.        ,  0.        , ..., -1.23101413,
        -1.61964375, -1.06088303],
       [ 1.        ,  1.        ,  0.        , ..., -0.42844371,
        -1.19737513, -0.64762079],
       [ 1.        ,  0.        ,  1.        , ..., -1.01981139,
         1.33623654,  1.5564445 ],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -1.69566016,
         1.33623654, -0.92312895],
       [ 1.        ,  1.        ,  0.        , ..., -1.06205194,
        -0.63435032,  1.00542818],
       [ 1.        ,  1.        ,  0.        , ..., -1.27325468,
        -0.0713255 ,  1.00542818]], shape=(780, 25))

In [72]:
pd.DataFrame(X_train)

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,-0.989151,0.247405,0.913968,-0.785375
1,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,-0.265845,0.036202,-0.634350,1.005428
2,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.443789,1.049975,0.913968,-1.474145
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,-1.646702,-0.977571,0.210187,-0.303236
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.917748,-0.343963,-0.915863,-0.923129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3115,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.509544,-0.808609,0.210187,0.316658
3116,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,-1.120661,0.162924,1.054724,-0.303236
3117,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,-1.186416,1.134457,0.491699,-1.543022
3118,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.128686,0.205165,0.632456,0.523289


## Training

In [94]:
from sklearn.ensemble  import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,classification_report,\
    precision_score,recall_score,f1_score,accuracy_score

In [95]:
## best way of model training 
models ={
    'logistic Regression':LogisticRegression(),
    'Decision Tree':DecisionTreeClassifier(),
    'Random Forest':RandomForestClassifier(),
    'KNN':KNeighborsClassifier(),
    'Naive Bayes':BernoulliNB()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)
    
    ## get the prediction 
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    ## performance on trianing data
    model_train_accuracy = accuracy_score(y_train,y_train_pred)
    model_train_precision = precision_score(y_train,y_train_pred,average='weighted')
    model_train_recall = recall_score(y_train,y_train_pred,average='weighted')
    model_train_f1 = f1_score(y_train,y_train_pred,average='weighted')
    
    
    ## performance on test data
    model_test_accuracy = accuracy_score(y_train,y_train_pred)
    model_test_precision = precision_score(y_train,y_train_pred,average='weighted')
    model_test_recall = recall_score(y_train,y_train_pred,average='weighted')
    model_test_f1 = f1_score(y_train,y_train_pred,average='weighted')
    
    
    ## To show the Name of the Model 
    print(list(models.values())[i])
    
    
    ### Displaying training performance 
    
    print('Accuracy{:.4f}'.format(model_train_accuracy))
    print('Precision {:4f}'.format(model_train_precision))
    print('Recall {:4f}'.format(model_train_precision))
    print('F1-Score {:4f}'.format(model_train_f1))
    
    print('-----------------------------------------------')
    
    ### Displaying test set performance 
    print('Accuracy{:.4f}'.format(model_test_accuracy))
    print('Precision {:4f}'.format(model_test_precision))
    print('Recall {:4f}'.format(model_test_precision))
    print('F1-Score {:4f}'.format(model_test_f1))
    
    
    print('='*35)
    print('\n')
    
    

LogisticRegression()
Accuracy0.2625
Precision 0.260359
Recall 0.260359
F1-Score 0.257759
-----------------------------------------------
Accuracy0.2625
Precision 0.260359
Recall 0.260359
F1-Score 0.257759


DecisionTreeClassifier()
Accuracy1.0000
Precision 1.000000
Recall 1.000000
F1-Score 1.000000
-----------------------------------------------
Accuracy1.0000
Precision 1.000000
Recall 1.000000
F1-Score 1.000000


RandomForestClassifier()
Accuracy1.0000
Precision 1.000000
Recall 1.000000
F1-Score 1.000000
-----------------------------------------------
Accuracy1.0000
Precision 1.000000
Recall 1.000000
F1-Score 1.000000


KNeighborsClassifier()
Accuracy0.3929
Precision 0.445925
Recall 0.445925
F1-Score 0.387475
-----------------------------------------------
Accuracy0.3929
Precision 0.445925
Recall 0.445925
F1-Score 0.387475


BernoulliNB()
Accuracy0.2545
Precision 0.253282
Recall 0.253282
F1-Score 0.247595
-----------------------------------------------
Accuracy0.2545
Precision 0.25328

## `RESULTS`
* `As you can see that Random Forest and Decision tree out perform all the ramaining Algorithms and giving 100% Accuracy`